In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [119]:
import pandas as pd
import numpy as np

In [120]:
df = pd.read_csv('/content/gdrive/MyDrive/Datathon/data/Train.csv')
# df = pd.read_csv('/content/gdrive/MyDrive/Datathon/data/Test.csv')   #uncomment while output

In [121]:
len(df)

11879

In [122]:
df.head()

,Id,Sex,Sex Code,State,State Code,Year,Year Code,Ten-Year Age Groups,Ten-Year Age Groups Code,% of Total Deaths,...,temp_sensor_readout,qc_flag_batch_3,legacy_index_offset,adjusted_pop_trend,confidence_spread_metric,Underlying_Cause,Contributing_Cause,Manner_of_Death,temporal_alignment_proxy,Target
0,0,F,F,Alabama,1.000000,2006.000000,NaN,< 1 year,1,0.000000,...,0.798753,0.514077,26,2.993754e+04,1.292495,1482,0.119917,3,async,NaN
1,1,F,F,Alabama,1.000000,2006.000000,2006.000000,1-4 years,1-4,0.000000,...,0.522892,-0.018962,19,NaN,NaN,6257,-0.047118,1,sync,NaN
2,2,F,F,Alabama,1.000000,2006.000000,NaN,5-14 years,5-14,0.000000,...,-0.131577,-1.147265,-44,3.066000e+05,1.728413,1171,0.051464,4,sync,55.0
3,3,NaN,NaN,Alabama,-91.432811,1980.124326,1980.124326,NaN,15-24,-0.079738,...,0.931934,0.214202,-24,2.849991e+06,NaN,6902,-0.073988,4,sync,NaN
4,4,F,F,Alabama,1.000000,2006.000000,2006.000000,25-34 years,25-34,0.000000,...,1.150727,-0.235712,22,3.013313e+05,1.243071,5361,0.190123,2,sync,327.0


In [127]:
df.isna().sum()

,0
Id,0
Sex,179
Sex Code,1453
State,1397
State Code,0
Year,4472
Year Code,4527
Ten-Year Age Groups,1500
Ten-Year Age Groups Code,1456
% of Total Deaths,0


# Data Cleaning Process

In [125]:
# using sex code to fill nan in sex
df['Sex'] = df['Sex'].fillna(df['Sex Code'])


In [128]:
df['Sex'].value_counts(dropna=False)

,count
Sex,
F,8249
M,3451
NaN,179


In [129]:
# fill null value with female as it is dominant
df['Sex'] = df['Sex'].fillna('F')

In [131]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['Sex'] = le.fit_transform(df['Sex'])
# Typically: F=0, M=1 (but check with le.classes_)

In [132]:
# deleting sex code
df.drop(columns=['Sex Code'], inplace=True)

In [133]:
df['State'].unique()

array(['Alabama', nan, 'Alaska', 'Arizona', 'Arkansas', 'California',
       'Colorado', 'Connecticut', 'Delaware', 'District of Columbia',
       'Florida', 'Georgia', 'Hawaii', 'Idaho', 'Illinois', 'Indiana',
       'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland',
       'Massachusetts', 'Michigan', 'Minnesota', 'Mississippi',
       'Missouri', 'Montana', 'Nebraska', 'Nevada', 'New Hampshire',
       'New Jersey', 'New Mexico', 'New York', 'North Carolina',
       'North Dakota', 'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania',
       'Rhode Island', 'South Carolina', 'South Dakota', 'Tennessee',
       'Texas', 'Utah', 'Vermont', 'Virginia', 'Washington',
       'West Virginia', 'Wisconsin', 'Wyoming'], dtype=object)

In [141]:
df['State Code'].unique()

array([  1.        , -91.43281059,  93.43281059,   2.        ,
        94.43281059, -90.43281059,   4.        , -88.43281059,
        96.43281059,   5.        , -87.43281059,  97.43281059,
         6.        ,  98.43281059, -86.43281059,   8.        ,
       -84.43281059, 100.43281059,   9.        , 101.43281059,
       -83.43281059,  10.        , -82.43281059, 102.43281059,
        11.        , -81.43281059, 103.43281059,  12.        ,
       104.43281059, -80.43281059,  13.        , -79.43281059,
       105.43281059,  15.        , -77.43281059, 107.43281059,
        16.        , 108.43281059, -76.43281059, 109.43281059,
        17.        , -75.43281059,  18.        , -74.43281059,
       110.43281059,  19.        , -73.43281059, 111.43281059,
        20.        , -72.43281059, 112.43281059,  21.        ,
       113.43281059, -71.43281059,  22.        , -70.43281059,
       114.43281059,  23.        , 115.43281059, -69.43281059,
        24.        , 116.43281059, -68.43281059,  25.  

In [142]:
# fill null of state  using state age combo features
df[['State_from_combo', 'AgeGroup_from_combo']] = (
    df['State_Age_Combo']
    .str.split('_', n=1, expand=True)
)


In [143]:
df['AgeGroup_from_combo'] = (
    df['AgeGroup_from_combo']
    .str.strip()
    .str.replace(' years', '', regex=False)
)

In [144]:
df['AgeGroup_from_combo'].value_counts()

,count
AgeGroup_from_combo,
35-44,1172
25-34,1168
45-54,1166
55-64,1160
85+,1154
65-74,1152
75-84,1145
< 1 year,1127
15-24,1112


In [146]:
df['State'] = df['State'].fillna(df['State_from_combo'])
df['Ten-Year Age Groups'] = df['Ten-Year Age Groups'].fillna(df['AgeGroup_from_combo'])


In [147]:
df.drop(columns=['State_from_combo'], inplace=True)
df.drop(columns=['AgeGroup_from_combo'], inplace=True)
df.drop(columns=['Ten-Year Age Groups Code'], inplace=True)

In [148]:
df.drop(columns=['State Code'], inplace=True)

In [149]:
df.isna().sum()

,0
Id,0
Sex,0
State,0
Year,4472
Year Code,4527
Ten-Year Age Groups,0
% of Total Deaths,0
Population,1386
Crude Rate,4506
Crude Rate Lower 95% Confidence Interval,0


In [150]:
# 1. Fill Year from Year Code
df['Year'] = df['Year'].fillna(df['Year Code'])

# 2. Force numeric
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')

# 3. Remove impossible years
# df.loc[~df['Year'].between(1900, 2025), 'Year'] = np.nan

# 4. TRUNCATE decimals (this is the key fix)
df['Year'] = np.floor(df['Year'])

# 5. Convert to nullable integer
df['Year'] = df['Year'].astype('Int64')

In [151]:
df.drop(columns=['Year Code'], inplace=True)

In [152]:
df.isna().sum()

,0
Id,0
Sex,0
State,0
Year,1752
Ten-Year Age Groups,0
% of Total Deaths,0
Population,1386
Crude Rate,4506
Crude Rate Lower 95% Confidence Interval,0
Crude Rate Upper 95% Confidence Interval,4593


In [153]:
df['Year'].fillna(df['Year'].median(), inplace=True)

/tmp/ipython-input-2633807203.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Year'].fillna(df['Year'].median(), inplace=True)


In [154]:
df['% of Total Deaths'] = pd.to_numeric(df['% of Total Deaths'], errors='coerce')
df.loc[df['% of Total Deaths'] < 0, '% of Total Deaths'] = 0
df['% of Total Deaths'] = df['% of Total Deaths'].clip(0, 1)

In [155]:
df.isna().sum()

,0
Id,0
Sex,0
State,0
Year,0
Ten-Year Age Groups,0
% of Total Deaths,0
Population,1386
Crude Rate,4506
Crude Rate Lower 95% Confidence Interval,0
Crude Rate Upper 95% Confidence Interval,4593


In [156]:
# Population
df['Population'] = pd.to_numeric(df['Population'], errors='coerce')
df['Population'] = df['Population'].fillna(df['adjusted_pop_trend'])
df['Population'] = df['Population'].fillna(df['Population'].median())


In [157]:

# Rates
rate_cols = [
    'Crude Rate',
    'Crude Rate Lower 95% Confidence Interval',
    'Crude Rate Upper 95% Confidence Interval',
    'Crude Rate Standard Error'
]

for col in rate_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.loc[df['Crude Rate'] < 0, 'Crude Rate'] = pd.NA
df.loc[df['Crude Rate Standard Error'] < 0, 'Crude Rate Standard Error'] = pd.NA

# Drop noisy CI columns
df = df.drop(columns=[
    'Crude Rate Lower 95% Confidence Interval',
    'Crude Rate Upper 95% Confidence Interval'
])


In [158]:
# Leave as-is for tree models
# OR, if needed:
df['Crude Rate'] = df['Crude Rate'].fillna(df['Crude Rate'].median())
df['Crude Rate Standard Error'] = df['Crude Rate Standard Error'].fillna(
    df['Crude Rate Standard Error'].median()
)

In [159]:
# deleting Year_dt features
df = df.drop(columns=['Year_dt'])


In [160]:
num_cols = [
    'temp_sensor_readout',
    'qc_flag_batch_3',
    'legacy_index_offset'
]

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')


In [ ]:
df = df.drop(columns=['adjusted_pop_trend'])
df = df.drop(columns=['confidence_spread_metric'])
df = df.drop(columns=['State_Age_Combo'])
df=df.drop(columns=['Id'])


In [163]:
df = df.drop(columns=['temporal_alignment_proxy'])     # comment while runnig test.csv

In [164]:
# Ensure numeric
num_cols = [
    'Population',
    'Contributing_Cause'
]
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Ensure categorical
cat_cols = [
    'Underlying_Cause',
    'Manner_of_Death',
    # 'temporal_alignment_proxy'
]
for col in cat_cols:
    df[col] = df[col].astype('category')

In [165]:
df.isna().sum()

,0
Sex,0
State,0
Year,0
Ten-Year Age Groups,0
% of Total Deaths,0
Population,0
Crude Rate,0
Crude Rate Standard Error,0
temp_sensor_readout,0
qc_flag_batch_3,0


# Using HistGradientBoostingRegressor

In [166]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

In [167]:
age_order = [
    '< 1 year', '1-4 years', '5-14 years', '15-24 years',
    '25-34 years', '35-44 years', '45-54 years',
    '55-64 years', '65-74 years', '75-84 years', '85+ years'
]

age_map = {age: i for i, age in enumerate(age_order)}
df['Ten-Year Age Groups'] = df['Ten-Year Age Groups'].map(age_map)


In [168]:
categorical_cols = [
    'State',
    'Underlying_Cause',
    'Manner_of_Death',
]

numeric_cols = [
    col for col in df.columns
    if col not in categorical_cols + ['Target']
]


In [169]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(
            handle_unknown='ignore',
            min_frequency=20,
            sparse_output=False # Added to ensure dense output
        ), categorical_cols),
        ('num', 'passthrough', numeric_cols)
    ]
)

In [170]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_score

train_df = df[df['Target'].notna()]
X = train_df.drop(columns=['Target'])
y = train_df['Target']

model = HistGradientBoostingRegressor(
    max_depth=6,
    learning_rate=0.05,
    max_iter=300,
    random_state=42
)

pipe = Pipeline([
    ('prep', preprocessor),
    ('model', model)
])

rmse = -cross_val_score(
    pipe, X, y,
    scoring='neg_root_mean_squared_error',
    cv=5
).mean()

print("CV RMSE:", rmse)


CV RMSE: 12538.57682083682


# using catboost

In [171]:
!pip install catboost

In [172]:
from catboost import CatBoostRegressor

In [173]:
cat_features = [
    'State',
    'Underlying_Cause',
    'Manner_of_Death',
]

train_df = df[df['Target'].notna()]
X = train_df.drop(columns=['Target'])
y = train_df['Target']

model = CatBoostRegressor(
    iterations=600,
    depth=6,
    learning_rate=0.05,
    loss_function='RMSE',
    random_seed=42,
    verbose=False,
    cat_features=cat_features # Pass cat_features directly to the constructor
)

rmse = -cross_val_score(
    model, X, y,
    scoring='neg_root_mean_squared_error',
    cv=5
).mean()

print("CV RMSE:", rmse)

CV RMSE: 12592.19049888341


In [174]:

# Compare with naive baselines
y_mean = np.mean(y)
y_median = np.median(y)

# 1. Mean predictor RMSE (worst-case naive model)
rmse_mean = np.sqrt(np.mean((y - y_mean) ** 2))

# 2. Median predictor RMSE
rmse_median = np.sqrt(np.mean((y - y_median) ** 2))

print(f"Mean of y: {y_mean:.2f}")
print(f"Std of y: {np.std(y):.2f}")
print(f"RMSE of predicting mean: {rmse_mean:.2f}")
print(f"Your model RMSE: {rmse:.2f}")
print(f"Model vs Mean predictor: {rmse/rmse_mean:.2%}")

Mean of y: 2345.64
Std of y: 14233.08
RMSE of predicting mean: 14233.08
Your model RMSE: 12592.19
Model vs Mean predictor: 88.47%


### *Creating Pipe to get output on test.csv . It little trick i have to run from start where i read test csv so i can apply feature engineering to test.csv data . then comes direct here not need to run above model building code*

In [175]:
# Load  only not null instance
train_df = df[df['Target'].notna()].copy()

### separate input and ouput
X_train = train_df.drop(columns=['Target'])
y_train = train_df['Target']

In [176]:
age_order = [
    '< 1 year', '1-4 years', '5-14 years', '15-24 years',
    '25-34 years', '35-44 years', '45-54 years',
    '55-64 years', '65-74 years', '75-84 years', '85+ years'
]

age_map = {age: i for i, age in enumerate(age_order)}

X_train['Ten-Year Age Groups'] = X_train['Ten-Year Age Groups'].map(age_map)


In [177]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor

categorical_cols = [
    'State',
    'Underlying_Cause',
    'Manner_of_Death'

]

numeric_cols = [
    col for col in X_train.columns
    if col not in categorical_cols
]

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(
            handle_unknown='ignore',
            min_frequency=20,
            sparse_output=False # Ensure dense output
        ), categorical_cols),
        ('num', 'passthrough', numeric_cols)
    ]
)

model = HistGradientBoostingRegressor(
    max_depth=6,
    learning_rate=0.05,
    max_iter=300,
    random_state=42
)

pipe = Pipeline([
    ('prep', preprocessor),
    ('model', model)
])

In [178]:
pipe.fit(X_train, y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                min_frequency=20,
                                                                sparse_output=False),
                                                  ['State', 'Underlying_Cause',
                                                   'Manner_of_Death']),
                                                 ('num', 'passthrough',
                                                  ['Sex', 'Year',
                                                   'Ten-Year Age Groups',
                                                   '% of Total Deaths',
                                                   'Population', 'Crude Rate',
                                                   'Crude Rate Standard Error',
                                                   'temp_sensor_readout',
                                                   'qc_flag_batch_3',
                                                   'legacy_index_offset',
                                                   'Contributing_Cause'])])),
                ('model',
                 HistGradientBoostingRegressor(learning_rate=0.05, max_depth=6,
                                               max_iter=300,
                                               random_state=42))])

Run below code while geting output

In [118]:
test_df  = pd.read_csv('/content/gdrive/MyDrive/Datathon/data/Test.csv') # to read id only

In [113]:
X_test = df.copy()

In [114]:
X_test['Ten-Year Age Groups']  = X_test['Ten-Year Age Groups'].map(age_map) #uncomment while test.csv

In [115]:
test_preds = pipe.predict(X_test)

In [116]:
submission_raw = pd.DataFrame({
    'Id': test_df['Id'],
    'Target': test_preds
})


In [117]:
submission = (
    submission_raw
    .groupby('Id', as_index=False)
    .agg({'Target': 'mean'})
)
print(submission['Id'].duplicated().sum() )  # MUST be 0
len(submission)                              # MUST be < 5092



0


4715